In [ ]:
"""
Tara's deepest darkest secrets
"""

__author__ = "Tara Powell"

import geopandas as gpd
import rasterio
import xarray as xr
import leafmap
import pandas as pd
import numpy as np
from pathlib import Path

# Tara Path
path = Path("/Users/tarapowell/Desktop/GIS AT3")

def f(r): 
    if r["Offence category"] == "Drug offences":
        return "Drug offences"
        
    elif r["Offence category"] == "Theft" or r["Offence category"] == "Arson" or r["Offence category"] == "Malicious damage to property":
        return "Property Offences"
    
    else:
        return "Violent Crime"


def createcatagories(file):
    inputfile = pd.read_csv( path / file ,sep = ',')
    inputfile.insert(1, "Crime Type", "")
    
    #https://www.geeksforgeeks.org/pandas/different-ways-to-iterate-over-rows-in-pandas-dataframe/
    dataframe = inputfile.copy()
    dataframe["Crime Type"] = dataframe.apply(f, axis=1)
    
    return dataframe


def split(dataframe):
    violentdf = dataframe[dataframe["Crime Type"] == "Violent Crime"]
    propertydf = dataframe[dataframe["Crime Type"] == "Property Offences"]
    drugdf = dataframe[dataframe["Crime Type"] == "Drug offences"]
    
    violentdf = violentdf.drop(columns = ["Crime Type"])
    propertydf = propertydf.drop(columns = ["Crime Type"])
    drugdf = drugdf.drop(columns = ["Crime Type"])
    
    return violentdf, propertydf, drugdf


def condenseyears(splitdataframe):
    dataframe = splitdataframe.copy()

    for i in range(15,26,1):
        dataframe[f"20{i}"] = dataframe[f"Jan-{i}"] + dataframe[f"Feb-{i}"] + dataframe[f"Mar-{i}"] + dataframe[f"Apr-{i}"] + dataframe[f"May-{i}"] + dataframe[f"Jun-{i}"] + dataframe[f"Jul-{i}"] + dataframe[f"Aug-{i}"] + dataframe[f"Sep-{i}"] + dataframe[f"Oct-{i}"] + dataframe[f"Nov-{i}"] + dataframe[f"Dec-{i}"]
        dataframe = dataframe.drop(columns = [f"Jan-{i}",f"Feb-{i}",f"Mar-{i}",f"Apr-{i}",f"May-{i}",f"Jun-{i}",f"Jul-{i}",f"Aug-{i}",f"Sep-{i}",f"Oct-{i}",f"Nov-{i}",f"Dec-{i}"])
    
    return dataframe


def condenseLGAs(splitdataframe):
    
    splitdataframe1 = splitdataframe.drop(columns = ["Offence category","Subcategory"])
    dataframe = splitdataframe1.groupby("LGA").sum()
    
    return dataframe


def main():
    dataframe = createcatagories("CrimeStatsbyLGA_Sydney.csv")
    
    violentdf, propertydf, drugdf = split(dataframe)
    
    violentdfcond1 = condenseyears(violentdf)
    propertydfcond1 = condenseyears(propertydf)
    drugdfcond1 = condenseyears(drugdf)
    
    violentdfcond1.to_csv(path / "ViolentCrime_LGA_ByCrime.csv", index=False)
    propertydfcond1.to_csv(path / "PropertyOffences_LGA_ByCrime.csv", index=False)
    drugdfcond1.to_csv(path / "Drugs_LGA_ByCrime.csv", index=False)
    
    violentdfcond2 = condenseLGAs(violentdfcond1)
    propertydfcond2 = condenseLGAs(propertydfcond1)
    drugdfcond2 = condenseLGAs(drugdfcond1)
    
    violentdfcond2.to_csv(path / "ViolentCrime_LGA_Totals.csv")
    propertydfcond2.to_csv(path / "PropertyOffences_LGA_Totals.csv")
    drugdfcond2.to_csv(path / "Drugs_LGA_Totals.csv")
    
    
if __name__ == "__main__":
    main()


KeyError: 'Jan-15'